In [ ]:
import pandas as pd
import re

from datasets import Dataset, load_dataset

### Load Datasets

In [ ]:
# Set up dataset stream
ocr_page_ds = load_dataset("RevolutionCrossroads/RevX_OCR_results",
                           name="page_level",
                           split="train",
                           streaming=True)

ocr_block_ds = load_dataset("RevolutionCrossroads/RevX_OCR_results",
                            name="block_level",
                            split="train",
                            streaming=True)

ocr_page_ds.features, ocr_block_ds.features

### Detect "price currents" in pages

In [ ]:
# Detect "price currents" in page
pc_re = re.compile(r"\bprices? currents?\b", re.IGNORECASE)

def page_has_price_currents(col, txt):
  return col == "chronicling_america" and txt and pc_re.search(txt.lower())

pc_pages_ds = ocr_page_ds.filter(page_has_price_currents, input_columns=["collection", "new_ocr_text"])

# pc_pages_ds.push_to_hub("visualizedata/price-currents-pages")

In [ ]:
# load "price currents" pages
pc_page_ds = load_dataset("visualizedata/price-currents-pages", split="train")
pc_pages = set(pc_page_ds["filename"])
len(pc_pages)

### Find html tables in "price currents" pages

In [ ]:
# Find blocks in the "price currents" pages that have html tables
tab_re = re.compile(r"<table\b", re.IGNORECASE)

def block_has_table(bfname, html):
  return bfname in pc_pages and tab_re.search(html.lower())

table_blocks_ds = ocr_block_ds.filter(block_has_table, input_columns=["filename", "block_html"])

def generate():
  yield from table_blocks_ds

table_blocks_eager_ds = Dataset.from_generator(generate)

# table_blocks_eager_ds.push_to_hub("visualizedata/table-blocks")

In [ ]:
# load table blocks
pc_tables_ds = load_dataset("visualizedata/table-blocks", split="train")
len(pc_tables_ds)

for block in pc_tables_ds:
  print(block["filename"])
  print(block["block_markdown"])

### Detect "price currents" in blocks

Independent of heading, and not in tables

In [ ]:
# Detect "price currents" in blocks
pc_re = re.compile(r"\bprices? currents?\b", re.IGNORECASE)

def block_has_price_currents(col, txt):
  return col == "chronicling_america" and txt and pc_re.search(txt.lower())

pc_blocks_ds = ocr_block_ds.filter(block_has_price_currents, input_columns=["collection", "block_text"])

def generate():
  yield from pc_blocks_ds

pc_blocks_eager_ds = Dataset.from_generator(generate)

pc_blocks_eager_ds.push_to_hub("visualizedata/price-currents-blocks")

In [ ]:
# load "price currents" blocks
pc_blocks_ds = load_dataset("visualizedata/price-currents-blocks", split="train")
print(len(pc_blocks_ds))

for block in pc_blocks_ds.take(200):
  print(block["filename"])
  print(block["block_text"])

In [ ]:
# TODO: use LLM to parse tables into structured dataset
# TODO: use filename to get paper name and date